In [ ]:
import json
import random
import string
import smtplib
from datetime import datetime
from email.mime.text import MIMEText

In [ ]:
def send_otp(receiver_email):

    otp = random.randint(1000, 9999)

    sender_email = "jaasmibank@gmail.com"

    sender_password = "lqwvpyzogicuojov"

    subject = "JASMI BANK"

    body = f"""

Welcome to JASMI BANK.

Dear Customer,

Your One-Time Password (OTP) for account recovery is:

{otp}

Please do not share this OTP with anyone.

Thank you for banking with us.

Regards,
JASMI BANK
Customer Support

"""

    message = MIMEText(body)

    message["Subject"] = subject
    message["From"] = sender_email
    message["To"] = receiver_email

    try:

        server = smtplib.SMTP("smtp.gmail.com", 587)

        server.starttls()

        server.login(sender_email, sender_password)

        server.sendmail(
            sender_email,
            receiver_email,
            message.as_string()
        )

        server.quit()

        print("OTP Sent Successfully!")

        return otp

    except Exception as e:

        print("Error:", e)

        return None

In [ ]:
try:

    with open("users.json", "r") as file:

        users = json.load(file)

except FileNotFoundError:

    users = {}


# Save User Data
def save_data():

    with open("users.json", "w") as file:

        json.dump(users, file, indent=4)

In [ ]:
def save_transaction(transaction):

    try:

        with open("transactions.json", "r") as file:

            transactions = json.load(file)

    except FileNotFoundError:

        transactions = []

    transactions.append(transaction)

    with open("transactions.json", "w") as file:

        json.dump(transactions, file, indent=4)

In [ ]:
def register():

    uname = input("Enter username: ")
    pwd = input("Enter password: ")

    if uname in users:

        print("User already exists!")

    else:

        email = input("Enter Email ID: ")

        # 3 random letters
        letters = ''.join(random.choices(string.ascii_uppercase, k=3))

        # 10 random numbers
        numbers = ''.join(random.choices(string.digits, k=10))

        # Account Number
        acc_no = letters + numbers

        # Create PIN
        pin = int(input("Create 4-digit PIN: "))
        confirm_pin = int(input("Confirm PIN: "))

        if pin != confirm_pin:

            print("PIN does not match!")
            return

        users[uname] = {

            "password": pwd,
            "email": email,
            "pin": pin,
            "locked": False,
            "balance": 0,
            "account_number": acc_no,
            "transactions": []

        }

        save_data()

        print("\nRegistration successful!")
        print("Account Number :", acc_no)

In [ ]:
def login():

    count = 3

    while count > 0:

        uname = input("Enter username: ")
        pwd = input("Enter password: ")

        if uname in users:

            if users[uname]["locked"]:

                print("Your account is locked!")
                return None

            elif users[uname]["password"] == pwd:

                print("Login successful!")
                return uname

            else:

                count -= 1
                print("Invalid password! Attempts left:", count)

        else:

            count -= 1
            print("Invalid username! Attempts left:", count)

    print("Too many failed attempts!")
    return None

In [ ]:
def verify_pin(uname):

    chances = 3

    while chances > 0:

        entered_pin = int(input("Enter PIN: "))

        if entered_pin == users[uname]["pin"]:

            return True

        else:

            chances -= 1

            print("Wrong PIN! Chances left:", chances)

    users[uname]["locked"] = True

    save_data()

    print("Account Locked!")

    return False

In [ ]:
def dashboard(uname):

    while True:

        print("\n===== DASHBOARD =====")
        print("1. Check Balance")
        print("2. Deposit")
        print("3. Withdraw")
        print("4. Transfer Money")
        print("5. Transactions")
        print("6. Logout")

        choice = input("Enter choice: ")

        # Check Balance
        if choice == "1":

            if verify_pin(uname):

                print("Account Number:", users[uname]["account_number"])
                print("Balance:", users[uname]["balance"])

            else:

                print("PIN verification failed!")

        # Deposit
        elif choice == "2":

            try:

                amt = float(input("Enter amount: "))

                if amt <= 0:

                    print("Enter valid amount!")

                else:

                    current_time = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

                    users[uname]["balance"] += amt

                    txn_id = random.randint(1000, 9999)

                    transaction = {

                        "txn_id": txn_id,
                        "account_number": users[uname]["account_number"],
                        "date_time": current_time,
                        "type": "Deposit",
                        "amount": amt

                    }

                    save_transaction(transaction)

                    users[uname]["transactions"].append(
                        f"Txn ID: {txn_id} | Acc No: {users[uname]['account_number']} | {current_time} | Deposited {amt}"
                    )

                    save_data()

                    print("Money deposited successfully!")

            except ValueError:

                print("Invalid input!")

        # Withdraw
        elif choice == "3":

            if not verify_pin(uname):

                print("PIN verification failed!")
                continue

            try:

                amt = float(input("Enter amount: "))

                if amt <= 0:

                    print("Enter valid amount!")

                elif users[uname]["balance"] >= amt:

                    current_time = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

                    users[uname]["balance"] -= amt

                    txn_id = random.randint(1000, 9999)

                    transaction = {

                        "txn_id": txn_id,
                        "account_number": users[uname]["account_number"],
                        "date_time": current_time,
                        "type": "Withdraw",
                        "amount": amt

                    }

                    save_transaction(transaction)

                    users[uname]["transactions"].append(
                        f"Txn ID: {txn_id} | Acc No: {users[uname]['account_number']} | {current_time} | Withdrawn {amt}"
                    )

                    save_data()

                    print("Money withdrawn successfully!")

                else:

                    print("Insufficient balance!")

            except ValueError:

                print("Invalid input!")

        # Transfer Money
        elif choice == "4":

            receiver_acc = input("Enter Receiver Account Number: ")

            found = False

            for user in users:

                if users[user]["account_number"] == receiver_acc:

                    found = True

                    if not verify_pin(uname):

                        print("PIN verification failed!")
                        break

                    amt = float(input("Enter amount to transfer: "))

                    if amt <= 0:

                        print("Enter valid amount!")

                    elif users[uname]["balance"] >= amt:

                        users[uname]["balance"] -= amt

                        users[user]["balance"] += amt

                        current_time = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

                        txn_id = random.randint(1000, 9999)

                        users[uname]["transactions"].append(
                            f"Txn ID: {txn_id} | Sent {amt} to {receiver_acc} | {current_time}"
                        )

                        users[user]["transactions"].append(
                            f"Txn ID: {txn_id} | Received {amt} from {users[uname]['account_number']} | {current_time}"
                        )

                        save_data()

                        print("Money transferred successfully!")

                    else:

                        print("Insufficient balance!")

                    break

            if found == False:

                print("Account Number not found!")

        # Transactions
        elif choice == "5":

            if not users[uname]["transactions"]:

                print("No transactions found!")

            else:

                print("\n--- Transactions ---")

                for t in users[uname]["transactions"]:

                    print(t)

        # Logout
        elif choice == "6":

            print("Logged out!")
            break

        # Invalid Choice
        else:

            print("Invalid choice!")

In [ ]:
def save_transaction(transaction):

    try:

        with open("transactions.json", "r") as file:

            transactions = json.load(file)

    except FileNotFoundError:

        transactions = []

    transactions.append(transaction)

    with open("transactions.json", "w") as file:

        json.dump(transactions, file, indent=4)

In [ ]:
def unlock_account():

    uname = input("Enter username: ")

    if uname not in users:

        print("User not found!")
        return

    receiver_email = users[uname]["email"]

    otp = send_otp(receiver_email)

    if otp == None:

        return

    user_otp = int(input("Enter OTP: "))

    if user_otp == otp:

        new_pin = int(input("Create New PIN: "))
        confirm_pin = int(input("Confirm New PIN: "))

        if new_pin == confirm_pin:

            users[uname]["pin"] = new_pin

            users[uname]["locked"] = False

            save_data()

            print("Account Unlocked Successfully!")
            print("New PIN Created!")

        else:

            print("PIN does not match!")

    else:

        print("Invalid OTP!")

In [ ]:
while True:

    print("\n1. Login")
    print("2. Register")
    print("3. Unlock Account")
    print("4. Exit")

    choice = input("Enter choice: ")

    if choice == "1":

        user = login()

        if user:

            dashboard(user)

    elif choice == "2":

        register()

    elif choice == "3":

        unlock_account()

    elif choice == "4":

        print("Goodbye!")
        break

    else:

        print("Invalid choice!")